In [ ]:
import re
import numpy as np
import unicodedata
import torch
from torch import nn
import torch.nn.functional as F
from collections import Counter
import os, unicodedata
train_on_gpu = torch.cuda.is_available()

In [ ]:
# # ---------- 1. Tải dữ liệu ----------
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_DIR = '/content/drive/MyDrive/char_rnn'          # thư mục trên Drive
# os.makedirs(DRIVE_DIR, exist_ok=True)
# !pip install -q pypdf
# import re, unicodedata
# from pypdf import PdfReader

# pdf_path = '/content/drive/MyDrive/char_rnn/sach.pdf'   # file PDF của bạn
# txt_path = '/content/drive/MyDrive/char_rnn/sach.txt'

# reader = PdfReader(pdf_path)
# text = '\n'.join(p.extract_text() or '' for p in reader.pages)

# text = unicodedata.normalize('NFC', text)                 # chuẩn hóa dấu tiếng Việt
# text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.M)     # bỏ dòng chỉ có số trang
# text = re.sub(r'-\n(\w)', r'\1', text)                  # nối từ bị ngắt gạch nối cuối dòng
# text = re.sub(r'\n{3,}', '\n\n', text)                  # gộp dòng trống thừa

# with open(DATA_PATH, 'r', encoding='utf-8') as f:
#     text = f.read()
# text = unicodedata.normalize('NFC', text)              # chuẩn hóa dấu (tiếng Việt)
# DATA_PATH = os.path.join(DRIVE_DIR, 'sach.txt')        # đổi tên file dữ liệu của bạn
# SAVE_PATH = os.path.join(DRIVE_DIR, 'word_rnn_best.net')

In [ ]:
!pip install -q pypdf
import os, re, unicodedata
from pypdf import PdfReader
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/char_rnn'
os.makedirs(DRIVE_DIR, exist_ok=True)
PDF_PATH  = os.path.join(DRIVE_DIR, 'sach.pdf')       # file PDF của bạn
DATA_PATH = os.path.join(DRIVE_DIR, 'sach.txt')       # file txt sẽ tạo ra
SAVE_PATH = os.path.join(DRIVE_DIR, 'word_rnn_best.net')

if not os.path.exists(DATA_PATH):                     # chỉ chuyển 1 lần
    reader = PdfReader(PDF_PATH)
    raw = '\n'.join(p.extract_text() or '' for p in reader.pages)
    raw = unicodedata.normalize('NFC', raw)
    raw = re.sub(r'^\s*\d+\s*$', '', raw, flags=re.M)   # bỏ dòng số trang
    raw = re.sub(r'-\n(\w)', r'\1', raw)                # nối từ bị ngắt
    raw = re.sub(r'\n{3,}', '\n\n', raw)                # gộp dòng trống
    with open(DATA_PATH, 'w', encoding='utf-8') as f:
        f.write(raw)
    print('Đã tạo', DATA_PATH)

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    text = f.read()
text = unicodedata.normalize('NFC', text)
print(len(text), 'ký tự')
print(text[:300])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
470121 ký tự

Mục lục
Giới Thiệu
Chương 1: Thư Hồi Âm Để Trong Hộp Nhận Sữa - (1)
(2)
(3)
(4)
(5)
(6)
Chương 2: Thổi Kèn Harmonica Giữa Đêm Khuya - (1)
(2)
(3)
(4)
(5)
(6)
(7)
(8)
(9)
(10)
Chương 3: Ở Trong Xe Civic Đến Sáng - (1)
(2)
(3)
(4)
(5)
(6)
(7)
Chương 4: Mặc Niệm Bằng Nhạc Beatles - (1)
(2)
(3)
(4)
(5)


In [ ]:
# ---------- 2. Tokenize theo từ ----------
TOKEN_RE = r"\w+|[^\w\s]|\n"
tokens = re.findall(TOKEN_RE, text.lower())
cnt = Counter(tokens)
vocab = ['<unk>'] + sorted(w for w, c in cnt.items() if c >= 3)   # sorted: thứ tự cố định
word2int = {w: i for i, w in enumerate(vocab)}
int2word = dict(enumerate(vocab))
encoded = np.array([word2int.get(w, 0) for w in tokens])
print('Số token:', len(tokens), '| Kích thước từ điển:', len(vocab))

Số token: 131445 | Kích thước từ điển: 1971


In [ ]:
print(tokens[:20])
print(encoded[:20])
print(' '.join(int2word[i] for i in encoded[:20]))     # giải mã ngược để kiểm tra

['\n', 'mục', 'lục', '\n', 'giới', 'thiệu', '\n', 'chương', '1', ':', 'thư', 'hồi', 'âm', 'để', 'trong', 'hộp', 'nhận', 'sữa', '-', '(']
[   1  937  835    1  467 1372    1  204   11   35 1413  604 1801 1909
 1500  610 1059 1336    8    5]

 mục lục 
 giới thiệu 
 chương 1 : thư hồi âm để trong hộp nhận sữa - (


In [ ]:
# ---------- 3. Mini-batch (giữ nguyên như bản ký tự) ----------
def get_batches(arr, batch_size, seq_length):
    batch_size_total = batch_size * seq_length
    n_batches = len(arr) // batch_size_total
    arr = arr[:n_batches * batch_size_total]
    arr = arr.reshape((batch_size, -1))
    for n in range(0, arr.shape[1], seq_length):
        x = arr[:, n:n+seq_length]
        y = np.zeros_like(x)
        try:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, n+seq_length]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, 0]
        yield x, y

train_on_gpu = torch.cuda.is_available()

In [ ]:
# ---------- 4. Mô hình: Embedding thay one-hot ----------
class WordRNN(nn.Module):
    def __init__(self, vocab, emb_dim=256, n_hidden=512, n_layers=2, drop_prob=0.5):
        super().__init__()
        self.vocab, self.n_hidden, self.n_layers = vocab, n_hidden, n_layers
        self.emb_dim, self.drop_prob = emb_dim, drop_prob
        self.embed = nn.Embedding(len(vocab), emb_dim)
        self.lstm = nn.LSTM(emb_dim, n_hidden, n_layers,
                            dropout=drop_prob, batch_first=True)
        self.dropout = nn.Dropout(drop_prob)
        self.fc = nn.Linear(n_hidden, len(vocab))

    def forward(self, x, hidden):
        out, hidden = self.lstm(self.embed(x), hidden)   # (batch, seq, n_hidden)
        out = self.dropout(out).contiguous().view(-1, self.n_hidden)
        return self.fc(out), hidden                      # (batch*seq, len(vocab))

    def init_hidden(self, batch_size):
        w = next(self.parameters())
        return (w.new_zeros(self.n_layers, batch_size, self.n_hidden),
                w.new_zeros(self.n_layers, batch_size, self.n_hidden))

In [ ]:
x, _ = next(get_batches(encoded, 8, 10))
test_net = WordRNN(vocab, emb_dim=32, n_hidden=64, n_layers=2)
h = test_net.init_hidden(8)
out, h = test_net(torch.from_numpy(x).long(), h)
print('output:', out.shape)                            # (8*10, len(vocab))
print('h, c:', h[0].shape, h[1].shape)                 # (2, 8, 64)

output: torch.Size([80, 1971])
h, c: torch.Size([2, 8, 64]) torch.Size([2, 8, 64])


In [ ]:
# ---------- 5. Huấn luyện (lưu lịch sử loss + model tốt nhất) ----------
def train(net, data, epochs=10, batch_size=64, seq_length=70, lr=0.001,
          clip=5, val_frac=0.1, print_every=50):
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    val_idx = int(len(data) * (1 - val_frac))
    data, val_data = data[:val_idx], data[val_idx:]
    if train_on_gpu:
        net.cuda()
    hist, best, counter = {'train': [], 'val': []}, float('inf'), 0
    for e in range(epochs):
        net.train()
        h = net.init_hidden(batch_size)
        for x, y in get_batches(data, batch_size, seq_length):
            counter += 1
            inputs, targets = torch.from_numpy(x).long(), torch.from_numpy(y).long()
            if train_on_gpu:
                inputs, targets = inputs.cuda(), targets.cuda()
            h = tuple(each.data for each in h)
            net.zero_grad()
            output, h = net(inputs, h)
            loss = criterion(output, targets.view(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), clip)
            opt.step()

            if counter % print_every == 0:
                net.eval()
                val_h, val_losses = net.init_hidden(batch_size), []
                with torch.no_grad():
                    for x, y in get_batches(val_data, batch_size, seq_length):
                        inputs, targets = torch.from_numpy(x).long(), torch.from_numpy(y).long()
                        if train_on_gpu:
                            inputs, targets = inputs.cuda(), targets.cuda()
                        output, val_h = net(inputs, val_h)
                        val_losses.append(criterion(output, targets.view(-1)).item())
                net.train()
                val = np.mean(val_losses)
                hist['train'].append(loss.item()); hist['val'].append(val)
                if val < best:                       # lưu model tốt nhất
                    best = val
                    torch.save({'state_dict': net.state_dict(), 'vocab': net.vocab,
                                'emb_dim': net.emb_dim, 'n_hidden': net.n_hidden,
                                'n_layers': net.n_layers}, '/content/drive/MyDrive/word_rnn_vi_best.net')
                print(f'Epoch {e+1}/{epochs} | Step {counter} | '
                      f'Loss {loss.item():.4f} | Val {val:.4f}')
    return hist

In [ ]:
# ---------- 6. Dự đoán và sinh văn bản theo từ ----------
def predict(net, word, h, top_k=5):
    x = torch.tensor([[word2int.get(word, 0)]])
    if train_on_gpu:
        x = x.cuda()
    h = tuple(each.data for each in h)
    out, h = net(x, h)
    p = F.softmax(out, dim=1).data.cpu()
    p[0, 0] = 0                                  # không sinh <unk>
    p, top = p.topk(top_k)
    p, top = p.numpy().squeeze(), top.numpy().squeeze()
    return int2word[np.random.choice(top, p=p/p.sum())], h

def detokenize(words):
    s = ' '.join(words)
    s = re.sub(r" ([.,;:!?'\"\n])", r"\1", s)
    return s.replace('\n ', '\n')

def sample(net, max_words=200, prime='anna said', top_k=5, min_words=10):
    prime = unicodedata.normalize('NFC', prime)
    net.cuda() if train_on_gpu else net.cpu()
    net.eval()
    prime = unicodedata.normalize('NFC', prime)
    words = re.findall(TOKEN_RE, prime.lower())
    h = net.init_hidden(1)
    with torch.no_grad():
        for w in words:
            nxt, h = predict(net, w, h, top_k)
        words.append(nxt)
        for i in range(max_words):
            if i >= min_words and words[-1] in ('.', '!', '?'):   # dừng ở cuối câu
                break
            nxt, h = predict(net, words[-1], h, top_k)
            words.append(nxt)
    return detokenize(words)

In [ ]:
net = WordRNN(vocab, emb_dim=256, n_hidden=512, n_layers=2)
print(net)

WordRNN(
  (embed): Embedding(1971, 256)
  (lstm): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=512, out_features=1971, bias=True)
)


In [ ]:
# hist = train(net, encoded, epochs=10, batch_size=64, seq_length=50, lr=0.001)
hist = train(net, encoded, epochs=30, batch_size=32, seq_length=30, lr=0.002, print_every=20)

Epoch 1/30 | Step 20 | Loss 6.0192 | Val 6.0638
Epoch 1/30 | Step 40 | Loss 5.8111 | Val 5.9363
Epoch 1/30 | Step 60 | Loss 5.7245 | Val 5.8586
Epoch 1/30 | Step 80 | Loss 5.5287 | Val 5.8201
Epoch 1/30 | Step 100 | Loss 5.5371 | Val 5.7398
Epoch 1/30 | Step 120 | Loss 5.4743 | Val 5.6868
Epoch 2/30 | Step 140 | Loss 5.2446 | Val 5.6023
Epoch 2/30 | Step 160 | Loss 5.3481 | Val 5.5468
Epoch 2/30 | Step 180 | Loss 5.3285 | Val 5.4918
Epoch 2/30 | Step 200 | Loss 5.1199 | Val 5.4552
Epoch 2/30 | Step 220 | Loss 5.1064 | Val 5.3903
Epoch 2/30 | Step 240 | Loss 4.9917 | Val 5.3665
Epoch 3/30 | Step 260 | Loss 5.0184 | Val 5.2946
Epoch 3/30 | Step 280 | Loss 5.0125 | Val 5.2719
Epoch 3/30 | Step 300 | Loss 4.8201 | Val 5.2261
Epoch 3/30 | Step 320 | Loss 4.8400 | Val 5.1808
Epoch 3/30 | Step 340 | Loss 4.9575 | Val 5.1235
Epoch 3/30 | Step 360 | Loss 4.5617 | Val 5.1204
Epoch 4/30 | Step 380 | Loss 4.7854 | Val 5.0809
Epoch 4/30 | Step 400 | Loss 4.5423 | Val 5.0522
Epoch 4/30 | Step 420 | 

In [ ]:
np.random.seed(0)                                     # bỏ dòng này nếu muốn mỗi lần ra khác nhau
print(sample(net, prime='giải thưởng', top_k=5))

anna said,
anh đã gọi điện cho bà hideyo. ông takeo đang nằm trong giường anh.


In [ ]:
ck = torch.load(SAVE_PATH, map_location='cuda' if train_on_gpu else 'cpu')
vocab = ck['vocab']
word2int = {w: i for i, w in enumerate(vocab)}
int2word = dict(enumerate(vocab))
loaded = WordRNN(vocab, ck['emb_dim'], ck['n_hidden'], ck['n_layers'])
loaded.load_state_dict(ck['state_dict'])

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/char_rnn/word_rnn_best.net'

In [ ]:
print(sample(loaded, prime='đại hội thể thao', top_k=5))